## Utilizzeremo il dataset "Quora query answers" per creare un modello in grado di identificare domande simili.

In [3]:
import pandas as pd
import numpy as np
import re
dataset = pd.read_csv('data/dataset.csv', index_col = 0)
dataset.head(5)

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0
3,3,7,8,Why am I mentally very lonely? How can I solve...,Find the remainder when [math]23^{24}[/math] i...,0
4,4,9,10,"Which one dissolve in water quikly sugar, salt...",Which fish would survive in salt water?,0


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
df = dataset.dropna(subset=['question1', 'question2'])

#df_positive = df[df['is_duplicate'] == 1].reset_index(drop=True)


# Tokenizzazione (lascio solo lettere e numeri)
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9]", " ", text)
    return text.split()

all_words = set()
for q in df['question1'].tolist() + df['question2'].tolist():
    all_words.update(clean_text(q))

vocab = {word: i + 1 for i, word in enumerate(all_words)}
vocab['<PAD>'] = 0
vocab['<UNK>'] = len(vocab)
vocab_size = len(vocab) + 1

In [5]:
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['is_duplicate']
)

print(f"Totale Train: {len(train_df)}")
print(f"Totale Test: {len(test_df)}")

# --- 2. DATASETS ---

# DATASET PER TRAINING (Solo Positivi per Triplet Loss)
train_df_positives = train_df[train_df['is_duplicate'] == 1].reset_index(drop=True)

Totale Train: 323478
Totale Test: 80870


## Defnisco il mio dataset: avrà solo anchor e positive

In [7]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import re
BATCH_SIZE = 16
MAX_LEN = 30
# Dataset Class
class TripletDataset(Dataset):
    def __init__(self, df, vocab, max_len=30):
        self.q1 = df['question1'].values
        self.q2 = df['question2'].values
        self.vocab = vocab
        self.max_len = max_len

    def encode(self, text):
        tokens = clean_text(text)
        seq = [self.vocab.get(t, self.vocab['<UNK>']) for t in tokens]
        if len(seq) < self.max_len:
            seq += [0] * (self.max_len - len(seq))
        else:
            seq = seq[:self.max_len]
        return seq

    def __len__(self):
        return len(self.q1)

    def __getitem__(self, idx):
        # Ritorniamo Anchor e Positive
        return torch.LongTensor(self.encode(self.q1[idx])), \
               torch.LongTensor(self.encode(self.q2[idx]))


# DATASET PER TEST (Con Label, per calcolare accuracy)
class EvaluationDataset(Dataset):
    def __init__(self, df, vocab):
        self.q1 = df['question1'].values
        self.q2 = df['question2'].values
        self.y = df['is_duplicate'].values # Ci serve la label vera!
        self.vocab = vocab
        # Riutilizziamo la logica di encoding
        self.encoder = TripletDataset(df, vocab)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        seq1 = self.encoder.encode(self.q1[idx])
        seq2 = self.encoder.encode(self.q2[idx])
        label = self.y[idx]
        return torch.LongTensor(seq1), torch.LongTensor(seq2), torch.tensor(label, dtype=torch.float)

train_ds = TripletDataset(train_df_positives, vocab)
test_ds = EvaluationDataset(test_df, vocab)               # Misto con label

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

## Defnisco la rete

In [16]:
class SiameseNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SiameseNet, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

    def forward_one(self, x):
        embeds = self.embedding(x)
        _, (hidden, _) = self.lstm(embeds)
        return hidden[-1] # Shape: [batch_size, hidden_dim] Prendo l'ultima attivazione

    def forward(self, x1, x2): #Applico due volte la rete all'input 1 e all input 2
        out1 = self.forward_one(x1)
        out2 = self.forward_one(x2)
        return out1, out2

## Definiamo la funzione di valutazione per l'accuratezza sul test set

In [9]:
def evaluate_model(model, loader, threshold=0.7):
    model.eval()
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for q1, q2, labels in loader:
            q1, q2 = q1.to(device), q2.to(device)

            emb1, emb2 = model(q1, q2)

            # Calcoliamo la Similarità Coseno
            similarities = F.cosine_similarity(emb1, emb2)

            all_scores.extend(similarities.cpu().numpy())
            all_labels.extend(labels.numpy())

    # Applichiamo la soglia: se sim > threshold -> 1, else -> 0
    predictions = [1 if score > threshold else 0 for score in all_scores]

    acc = accuracy_score(all_labels, predictions)
    f1 = f1_score(all_labels, predictions)

    return acc, f1, all_scores, all_labels

## Definiamo la logica per l'hard mining

In [13]:
import torch
import torch.nn.functional as F

def get_hard_triplets_cosine(anchor_embeddings, positive_embeddings):
    """
    Trova i negativi difficili usando la Cosine Similarity.
    Hard Negative = Il vettore (diverso dal positivo) che ha il coseno più alto con l'ancora.
    """
    # 1. Normalizziamo i vettori (L2 norm)
    # Cosi il prodotto matriciale diventa equivalente alla Cosine Similarity
    a_norm = F.normalize(anchor_embeddings, p=2, dim=1)
    p_norm = F.normalize(positive_embeddings, p=2, dim=1)

    # 2. Calcoliamo la matrice di similarità (Batch x Batch)
    # sim_matrix[i][j] è la similarità tra Anchor_i e Positive_j
    sim_matrix = torch.mm(a_norm, p_norm.t())

    # 3. Trasformiamo in Distanza Coseno (per riusare la logica del 'minimo')
    # Distanza = 1 - Similarità (0=identici, 2=opposti)
    dist_matrix = 1 - sim_matrix

    hard_negatives = []
    batch_size = dist_matrix.size(0)

    for i in range(batch_size):
        # Per l'ancora 'i', cerchiamo il negativo con la distanza minore
        # (ovvero la similarità maggiore), escludendo il positivo corretto 'i'

        current_dists = dist_matrix[i].clone()
        current_dists[i] = float('inf') # Mascheriamo il positivo corretto

        # Troviamo l'indice del negativo più "vicino"
        hard_neg_idx = torch.argmin(current_dists)

        # Prendiamo l'embedding originale (non normalizzato, se serve al modello)
        # o quello normalizzato. Di solito si usa quello grezzo se la loss lo gestisce,
        # ma qui prendiamo quello grezzo per coerenza col training loop.
        hard_negatives.append(positive_embeddings[hard_neg_idx])

    return torch.stack(hard_negatives)

# Defniamo la triplet loss

In [12]:
class TripletCosineLoss(nn.Module):
    def __init__(self, margin=0.5):
        super(TripletCosineLoss, self).__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        # Calcoliamo la similarità coseno per le coppie Positive e Negative
        # Dim=1 perché lavoriamo sui vettori (batch, hidden_dim)
        sim_pos = F.cosine_similarity(anchor, positive, dim=1)
        sim_neg = F.cosine_similarity(anchor, negative, dim=1)

        # Convertiamo in distanze (più piccolo = meglio)
        dist_pos = 1 - sim_pos
        dist_neg = 1 - sim_neg

        # Loss classica: vogliamo che dist_pos < dist_neg di almeno 'margin'
        losses = F.relu(dist_pos - dist_neg + self.margin)

        return losses.mean() #Faccio la media perchè ho un batch

In [14]:
def off_diagonal_loss(anchor, positive):
    # 1. Normalizziamo i vettori per avere la cosine similarity con il prodotto matriciale
    a_norm = F.normalize(anchor, p=2, dim=1)
    p_norm = F.normalize(positive, p=2, dim=1)

    # 2. Calcoliamo la Cross-Correlation Matrix (Batch x Batch)
    # Elemento [i, j] = Similarità tra Anchor i e Positive j
    sim_matrix = torch.mm(a_norm, p_norm.t())

    # 3. Creiamo una maschera per Rimuovere la diagonale
    # (La diagonale contiene le coppie corrette, che vogliamo alte, non basse!)
    batch_size = anchor.size(0)
    eye = torch.eye(batch_size, device=anchor.device)
    mask = 1 - eye # 0 sulla diagonale, 1 altrove

    # 4. Prendiamo solo i valori fuori diagonale
    off_diag_values = sim_matrix * mask

    # 5. Loss = Media dei Quadrati dei valori fuori diagonale
    # Dividiamo per il numero di elementi fuori diagonale (N^2 - N)
    n_off_diag = (batch_size * batch_size) - batch_size
    loss = (off_diag_values ** 2).sum() / n_off_diag

    return loss

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMBED_DIM = 50
HIDDEN_DIM = 64
MARGIN = 0.3 #
LAMBDA_OFF_DIAG = 0.1 # Peso della mean loss
model = SiameseNet(vocab_size, EMBED_DIM, HIDDEN_DIM).to(device)

criterion = TripletCosineLoss(margin=MARGIN)
optimizer = optim.Adam(model.parameters(), lr=0.0004)


print("Inizio Training (Triplet + Off-Diagonal Regularization)...")

for epoch in range(15):
    model.train()
    total_loss = 0
    loss_triplet_log = 0
    loss_off_log = 0

    for anchor_batch, positive_batch in train_loader:
        if anchor_batch.shape[0] < 2: continue #Se ho un solo elemento nel batch non posso fare hard mining

        anchor_batch = anchor_batch.to(device)
        positive_batch = positive_batch.to(device)

        optimizer.zero_grad()

        # 1. Forward Pass
        anchor_emb, positive_emb = model(anchor_batch, positive_batch)

        # 2. Hard Mining & Triplet Loss (Il compito principale)
        negative_emb = get_hard_triplets_cosine(anchor_emb, positive_emb)
        loss_main = criterion(anchor_emb, positive_emb, negative_emb)

        # 3. Off-Diagonal Loss (Il "Contorno" che pulisce lo spazio)
        loss_reg = off_diagonal_loss(anchor_emb, positive_emb)

        # 4. Somma Ponderata
        loss = loss_main + (LAMBDA_OFF_DIAG * loss_reg)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loss_triplet_log += loss_main.item()
        loss_off_log += loss_reg.item()

    avg_triplet = loss_triplet_log / len(train_loader)
    avg_off = loss_off_log / len(train_loader)
    # 2. EVALUATION (Sul Test Set completo)
    # Usiamo una soglia di default di 0.7
    val_acc, val_f1, _, _ = evaluate_model(model, test_loader, threshold=0.7)

    print(f"Epoch {epoch+1} | Total: {total_loss/len(train_loader):.4f} "
          f"(Triplet: {avg_triplet:.4f} | Off-Diag: {avg_off:.4f}) | "
          f"Test Acc: {val_acc*100:.2f}% | Test F1: {val_f1:.2f}")

Inizio Training (Triplet + Off-Diagonal Regularization)...
Epoch 1 | Total: 0.4001 (Triplet: 0.3002 | Off-Diag: 0.9985) | Test Acc: 36.92% | Test F1: 0.54
Epoch 2 | Total: 0.3971 (Triplet: 0.2986 | Off-Diag: 0.9848) | Test Acc: 37.55% | Test F1: 0.54
Epoch 3 | Total: 0.3959 (Triplet: 0.2981 | Off-Diag: 0.9788) | Test Acc: 38.68% | Test F1: 0.55


KeyboardInterrupt: 